# AnchorKV: standalone T4 experiment

Upload this single notebook to Google Colab, select a T4 GPU, and run all cells. It contains the bounded trace-extraction and receiver-head analysis code directly, so it does not clone or import the private GitHub repository. It saves compact sentence-level attention artifacts; it does **not** claim physical KV-cache compression or latency improvements.

In [ ]:
%pip install -q "transformers>=4.52.4" huggingface_hub matplotlib

In [ ]:
import torch

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} ({gpu_gib:.1f} GiB)')
if 'T4' not in gpu_name:
    print('Warning: this experiment was calibrated for a T4; record the actual GPU.')

In [ ]:
# Conservative lower-bound memory estimate for the bounded eager-attention replay.
parameters, layers, query_heads, kv_heads, head_dim = 600_000_000, 28, 16, 8, 128
sequence_length, dtype_bytes = 768, 2
weight_bytes = parameters * dtype_bytes
attention_bytes = layers * query_heads * sequence_length**2 * dtype_bytes
kv_bytes = layers * 2 * kv_heads * sequence_length * head_dim * dtype_bytes
estimate = {
    'model_weights_gib': round(weight_bytes / 1024**3, 4),
    'returned_attentions_gib': round(attention_bytes / 1024**3, 4),
    'kv_cache_gib': round(kv_bytes / 1024**3, 4),
    'lower_bound_gib': round((weight_bytes + attention_bytes + kv_bytes) / 1024**3, 4),
    'safe_limit_gib': round(gpu_gib * 0.8, 4),
}
estimate

In [ ]:
# Standalone AnchorKV helpers.
import gc
import hashlib
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

BOUNDARY = re.compile(r'(?<=[.!?])(?:\s+|$)|\n+')

def sentence_character_spans(text):
    spans, cursor = [], 0
    for match in BOUNDARY.finditer(text):
        start, end = cursor, match.start()
        while start < end and text[start].isspace(): start += 1
        while end > start and text[end - 1].isspace(): end -= 1
        if start < end: spans.append((start, end))
        cursor = match.end()
    start, end = cursor, len(text)
    while start < end and text[start].isspace(): start += 1
    while end > start and text[end - 1].isspace(): end -= 1
    if start < end: spans.append((start, end))
    return spans

def decoded_token_offsets(tokenizer, token_ids):
    token_ids = [int(token_id) for token_id in token_ids]
    text = tokenizer.decode(token_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    special_ids = set(tokenizer.all_special_ids)
    content_ids = [token_id for token_id in token_ids if token_id not in special_ids]
    encoded_ids = [int(token_id) for token_id in encoded['input_ids']]
    encoded_offsets = [tuple(map(int, pair)) for pair in encoded['offset_mapping']]
    if encoded_ids != content_ids or len(encoded_offsets) != len(content_ids):
        raise RuntimeError('Decoded text did not round-trip to the generated token IDs.')
    offsets, content_index = [], 0
    for token_id in token_ids:
        if token_id in special_ids:
            cursor = encoded_offsets[content_index][0] if content_index < len(encoded_offsets) else len(text)
            offsets.append((cursor, cursor))
        else:
            offsets.append(encoded_offsets[content_index])
            content_index += 1
    return text, offsets

def token_spans_from_offsets(text, offsets):
    result = []
    for char_start, char_end in sentence_character_spans(text):
        sentence_text = text[char_start:char_end]
        if sentence_text.strip() in {'<think>', '</think>'}: continue
        token_ids = [i for i, (start, end) in enumerate(offsets) if end > start and end > char_start and start < char_end]
        if token_ids:
            result.append((token_ids[0], token_ids[-1] + 1, sentence_text))
    return result

def reduce_attention_layers(attention_layers, spans, query_end=None):
    reduced = []
    for layer_attention in attention_layers:
        values = layer_attention[0].detach().float().cpu().numpy()
        effective_query_end = values.shape[-1] if query_end is None else query_end
        if not 0 < effective_query_end <= values.shape[-1]: raise ValueError('query_end is outside the attention sequence')
        sentence_scores = np.zeros((values.shape[0], len(spans)), dtype=np.float32)
        for sentence_index, (start, end, _) in enumerate(spans):
            if end < effective_query_end:
                future = values[:, end:effective_query_end, start:end]
                sentence_scores[:, sentence_index] = future.sum(axis=-1).mean(axis=-1) / (end - start)
        reduced.append(sentence_scores)
    return np.stack(reduced, axis=0)

def save_trace(path, metadata, spans, vertical_scores):
    destination = Path(path).with_suffix('.npz')
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f'.{destination.name}.tmp.npz')
    np.savez_compressed(
        temporary,
        vertical_scores=np.asarray(vertical_scores, dtype=np.float32),
        span_starts=np.asarray([span[0] for span in spans], dtype=np.int32),
        span_ends=np.asarray([span[1] for span in spans], dtype=np.int32),
        span_text=np.asarray([span[2] for span in spans], dtype=np.str_),
        metadata_json=np.asarray(json.dumps(metadata, sort_keys=True), dtype=np.str_),
    )
    os.replace(temporary, destination)
    return destination

def load_trace(path):
    with np.load(path, allow_pickle=False) as artifact:
        return {
            'metadata': json.loads(str(artifact['metadata_json'].item())),
            'vertical_scores': artifact['vertical_scores'].astype(np.float32),
            'span_starts': artifact['span_starts'].astype(np.int64),
            'span_ends': artifact['span_ends'].astype(np.int64),
            'span_text': artifact['span_text'].astype(np.str_),
        }

def trace_summary(trace):
    metadata, scores = trace['metadata'], trace['vertical_scores']
    return {
        'sample_id': metadata['sample_id'],
        'sequence_length': metadata['sequence_length'],
        'sentences': len(trace['span_starts']),
        'layers': metadata['layers'],
        'query_heads': metadata['query_heads'],
        'score_min': float(scores.min()),
        'score_max': float(scores.max()),
    }

def discover_receiver_heads(paths, top_k=16):
    traces = [load_trace(path) for path in paths]
    reference = traces[0]['metadata']
    per_trace = []
    for trace in traces:
        values = trace['vertical_scores'].astype(np.float64)
        centered = values - values.mean(axis=-1, keepdims=True)
        second, fourth = np.mean(centered**2, axis=-1), np.mean(centered**4, axis=-1)
        kurtosis = np.divide(fourth, second**2, out=np.zeros_like(fourth), where=second > 1e-12)
        per_trace.append(kurtosis)
    stacked = np.stack(per_trace, axis=0)
    means = stacked.mean(axis=0)
    percentile_ranks = []
    for values in per_trace:
        flat = values.ravel()
        _, inverse, counts = np.unique(flat, return_inverse=True, return_counts=True)
        starts = np.cumsum(counts) - counts
        midranks = starts + (counts - 1) / 2
        percentile_ranks.append((midranks[inverse] / (flat.size - 1)).reshape(values.shape))
    percentile_ranks = np.stack(percentile_ranks, axis=0)
    mean_percentiles = percentile_ranks.mean(axis=0)
    percentile_deviations = percentile_ranks.std(axis=0)
    stability = 1.0 / (1.0 + percentile_deviations / np.maximum(np.abs(mean_percentiles), 1e-12))
    heads = [
        {'layer': layer, 'query_head': head, 'mean_kurtosis': float(means[layer, head]),
         'mean_percentile': float(mean_percentiles[layer, head]),
         'stability': float(stability[layer, head]),
         'ranking_score': float(mean_percentiles[layer, head] * stability[layer, head])}
        for layer in range(means.shape[0]) for head in range(means.shape[1])
    ]
    heads.sort(key=lambda item: item['ranking_score'], reverse=True)
    heads = heads[:top_k]
    group_size = reference['query_heads'] // reference['kv_heads']
    kv_scores = {}
    for head in heads:
        key = (head['layer'], head['query_head'] // group_size)
        kv_scores[key] = max(kv_scores.get(key, float('-inf')), head['ranking_score'])
    return {
        'schema_version': 1, 'model_id': reference['model_id'],
        'model_revision': reference['model_revision'],
        'source_samples': [trace['metadata']['sample_id'] for trace in traces],
        'receiver_heads': heads,
        'kv_heads': [
            {'layer': key[0], 'kv_head': key[1], 'score': score}
            for key, score in sorted(kv_scores.items(), key=lambda item: item[1], reverse=True)
        ],
    }

def render_prompt(tokenizer, prompt):
    if not hasattr(tokenizer, 'apply_chat_template'): return prompt
    messages = [{'role': 'user', 'content': prompt}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def extract_trace(prompt, sample_id, output_path, model_id, model_revision, max_sequence_length=768, max_new_tokens=512, seed=7, min_reasoning_spans=6, min_future_tokens=32):
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=model_revision)
    rendered = render_prompt(tokenizer, prompt)
    inputs = tokenizer(rendered, return_tensors='pt')
    prompt_tokens = int(inputs['input_ids'].shape[-1])
    allowed_new_tokens = min(max_new_tokens, max_sequence_length - prompt_tokens)
    if allowed_new_tokens <= 0: raise ValueError('Prompt leaves no room under the trace limit.')
    model = AutoModelForCausalLM.from_pretrained(
        model_id, revision=model_revision, dtype=torch.float16, attn_implementation='sdpa'
    ).to('cuda').eval()
    device_inputs = {key: value.to('cuda') for key, value in inputs.items()}
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode():
        generated_ids = model.generate(
            **device_inputs, max_new_tokens=allowed_new_tokens, do_sample=False,
            use_cache=True, pad_token_id=tokenizer.eos_token_id,
        )
    full_ids = generated_ids[:, :max_sequence_length]
    generated_ids_list = full_ids[0, prompt_tokens:].tolist()
    generated_text, offsets = decoded_token_offsets(tokenizer, generated_ids_list)
    eos_token_ids = model.generation_config.eos_token_id
    eos_token_ids = {eos_token_ids} if isinstance(eos_token_ids, int) else set(eos_token_ids or [])
    ended_with_eos = bool(generated_ids_list and generated_ids_list[-1] in eos_token_ids)
    if len(generated_ids_list) >= allowed_new_tokens and not ended_with_eos:
        raise RuntimeError(f'Generation hit the {allowed_new_tokens}-token limit before EOS; do not use this truncated trace.')
    query_end = int(full_ids.shape[-1]) - int(ended_with_eos)
    local_spans = token_spans_from_offsets(generated_text, offsets)
    spans = [(start + prompt_tokens, end + prompt_tokens, text) for start, end, text in local_spans if end + prompt_tokens <= query_end - min_future_tokens]
    if not spans: raise RuntimeError('Generated trace contained no sentence-like steps.')
    if len(spans) < min_reasoning_spans:
        raise RuntimeError(f'Trace has only {len(spans)} reasoning spans; at least {min_reasoning_spans} are required for kurtosis.')
    if not hasattr(model, 'set_attn_implementation'):
        raise RuntimeError('Transformers cannot switch attention backends; restart and reinstall.')
    model.set_attn_implementation('eager')
    with torch.inference_mode():
        replay = model(
            input_ids=full_ids, attention_mask=torch.ones_like(full_ids),
            output_attentions=True, use_cache=False, return_dict=True,
        )
    if replay.attentions is None: raise RuntimeError('Model returned no eager attention tensors.')
    vertical_scores = reduce_attention_layers(replay.attentions, spans, query_end=query_end)
    config = model.config
    layers = int(config.num_hidden_layers)
    query_heads = int(config.num_attention_heads)
    kv_heads = int(getattr(config, 'num_key_value_heads', query_heads))
    head_dim = int(getattr(config, 'head_dim', config.hidden_size // query_heads))
    metadata = {
        'schema_version': 1, 'model_id': model_id, 'model_revision': model_revision,
        'sample_id': sample_id, 'prompt_sha256': hashlib.sha256(prompt.encode()).hexdigest(),
        'seed': seed, 'dtype': 'float16', 'sequence_length': int(full_ids.shape[-1]),
        'layers': layers, 'query_heads': query_heads, 'kv_heads': kv_heads,
        'head_dim': head_dim, 'created_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    artifact_path = save_trace(output_path, metadata, spans, vertical_scores)
    peak_gpu_bytes = int(torch.cuda.max_memory_allocated())
    del replay, model, device_inputs, generated_ids, full_ids
    gc.collect()
    torch.cuda.empty_cache()
    return {
        'artifact_path': artifact_path, 'generated_text': generated_text,
        'prompt_tokens': prompt_tokens, 'generated_tokens': len(generated_ids_list),
        'sequence_length': metadata['sequence_length'], 'peak_gpu_bytes': peak_gpu_bytes,
    }

In [ ]:
from huggingface_hub import model_info

MODEL_ID = 'Qwen/Qwen3-0.6B'
MODEL_REVISION = model_info(MODEL_ID).sha
MAX_SEQUENCE_LENGTH = 768
MAX_NEW_TOKENS = 512
MIN_REASONING_SPANS = 6
MIN_FUTURE_TOKENS = 32
SEED = 7
print('Pinned model revision:', MODEL_REVISION)

In [ ]:
PROMPTS = [
    "Reason step by step in at least 6 concise complete sentences and end with 'Answer: ...': If 3 notebooks cost $12, how much do 7 notebooks cost at the same rate?",
    "Reason step by step in at least 6 concise complete sentences and end with 'Answer: ...': A train travels 180 miles in 3 hours. At the same speed, how far does it travel in 5 hours?",
    "Reason step by step in at least 6 concise complete sentences and end with 'Answer: ...': Maria has twice as many marbles as Lee. Together they have 36 marbles. How many does each person have?",
]

In [ ]:
EXPECTED_ANSWER_TERMS = [('28',), ('300',), ('24', '12')]

In [ ]:
output_dir = Path('/content/anchorkv-artifacts')
output_dir.mkdir(parents=True, exist_ok=True)
artifact_paths, run_summaries = [], []
for index, (prompt, expected_terms) in enumerate(zip(PROMPTS, EXPECTED_ANSWER_TERMS, strict=True)):
    sample_id = f'math-{index:03d}'
    result = extract_trace(
        prompt, sample_id, output_dir / sample_id, MODEL_ID, MODEL_REVISION,
        max_sequence_length=MAX_SEQUENCE_LENGTH, max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
        min_reasoning_spans=MIN_REASONING_SPANS,
        min_future_tokens=MIN_FUTURE_TOKENS,
    )
    missing_terms = [term for term in expected_terms if term not in result['generated_text']]
    if missing_terms:
        raise RuntimeError(f"{sample_id} failed answer validation; missing {missing_terms}: {result['generated_text']}")
    artifact_paths.append(result['artifact_path'])
    summary = {key: value for key, value in result.items() if key not in {'artifact_path', 'generated_text'}}
    summary['sample_id'] = sample_id
    summary['peak_gpu_gib'] = result['peak_gpu_bytes'] / 1024**3
    summary['generated_text'] = result['generated_text']
    summary['answer_check'] = 'passed'
    run_summaries.append(summary)
    print(summary)

In [ ]:
import transformers

run_report = {
    'schema_version': 1,
    'gpu': {'name': gpu_name, 'total_gib': gpu_gib},
    'software': {'torch': torch.__version__, 'transformers': transformers.__version__},
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'config': {
        'max_sequence_length': MAX_SEQUENCE_LENGTH,
        'max_new_tokens': MAX_NEW_TOKENS,
        'seed': SEED,
        'enable_thinking': True,
        'min_reasoning_spans': MIN_REASONING_SPANS,
        'min_future_tokens': MIN_FUTURE_TOKENS,
    },
    'memory_estimate': estimate,
    'samples': run_summaries,
}
run_report_path = output_dir / 'run-summary.json'
run_report_path.write_text(json.dumps(run_report, indent=2, sort_keys=True) + '\n', encoding='utf-8')
run_report

In [ ]:
[trace_summary(load_trace(path)) for path in artifact_paths]

In [ ]:
manifest = discover_receiver_heads(artifact_paths, top_k=16)
manifest_path = output_dir / 'receiver-heads.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
manifest['receiver_heads'][:5]

In [ ]:
import matplotlib.pyplot as plt

top_heads = manifest['receiver_heads'][:10]
labels = [f"L{head['layer']}:H{head['query_head']}" for head in top_heads]
scores = [head['ranking_score'] for head in top_heads]
plt.figure(figsize=(10, 4))
plt.bar(labels, scores)
plt.ylabel('kurtosis × stability')
plt.title('AnchorKV receiver-head candidates')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(output_dir / 'receiver-heads.png', dpi=160)
plt.show()

## Pilot causal validation

The next cells suppress each eligible sentence across all attention heads during teacher-forced replay. Every intervention uses the same downstream token window. Raw receiver-head, cross-head-normalized, all-head, recency, and random selectors are evaluated against the causal oracle using rank correlation, top-k overlap, and regret. This tests sentence causality, not head-specific causality, and the three-prompt result remains a pilot rather than a final benchmark.

In [ ]:
import torch.nn.functional as F

def causal_mask_tensor(sequence_length, blocked_span=None, query_end=None, dtype=torch.float16):
    effective_query_end = sequence_length if query_end is None else query_end
    mask = torch.zeros((1, 1, sequence_length, sequence_length), device='cuda', dtype=dtype)
    future = torch.triu(torch.ones((sequence_length, sequence_length), device='cuda', dtype=torch.bool), diagonal=1)
    mask.masked_fill_(future[None, None], float('-inf'))
    if blocked_span is not None:
        start, end = blocked_span
        if end < effective_query_end:
            mask[:, :, end:effective_query_end, start:end] = float('-inf')
    return mask

def midranks(values):
    _, inverse, counts = np.unique(values, return_inverse=True, return_counts=True)
    starts = np.cumsum(counts) - counts
    return (starts + (counts - 1) / 2)[inverse].astype(np.float64)

def ranking_diagnostics(proxy_scores, causal_scores, candidate_indices=None, top_k=3):
    proxy_scores = np.asarray(proxy_scores, dtype=np.float64)
    causal_scores = np.asarray(causal_scores, dtype=np.float64)
    indices = np.arange(len(proxy_scores)) if candidate_indices is None else np.asarray(candidate_indices)
    if len(indices) < 2: raise RuntimeError('Ranking diagnostics require at least two candidates.')
    proxy, causal = proxy_scores[indices], causal_scores[indices]
    k = min(top_k, len(indices))
    proxy_order = np.lexsort((indices, -proxy))[:k]
    causal_order = np.lexsort((indices, -causal))[:k]
    proxy_top = indices[proxy_order]
    causal_top = indices[causal_order]
    proxy_rank, causal_rank = midranks(proxy), midranks(causal)
    proxy_centered = proxy_rank - proxy_rank.mean()
    causal_centered = causal_rank - causal_rank.mean()
    denominator = np.sqrt(np.sum(proxy_centered ** 2) * np.sum(causal_centered ** 2))
    return {
        'spearman': 0.0 if denominator <= 1e-12 else float(np.sum(proxy_centered * causal_centered) / denominator),
        'top_k': int(k),
        'top_k_overlap': float(len(set(proxy_top).intersection(causal_top)) / k),
        'top_k_regret': float(np.mean(causal_scores[causal_top]) - np.mean(causal_scores[proxy_top])),
        'top_1_regret': float(causal_scores[causal_top[0]] - causal_scores[proxy_top[0]]),
        'proxy_top_indices': [int(index) for index in proxy_top],
        'causal_top_indices': [int(index) for index in causal_top],
    }

def cross_head_normalized_scores(layer_scores, query_head):
    log_scores = np.log(np.maximum(np.asarray(layer_scores, dtype=np.float64), 1e-12))
    return log_scores[query_head] - np.median(log_scores, axis=0)

def intervention_metrics(reference_logits, intervention_logits, labels):
    reference_log_probs = F.log_softmax(reference_logits.float(), dim=-1)
    intervention_log_probs = F.log_softmax(intervention_logits.float(), dim=-1)
    divergences = torch.sum(reference_log_probs.exp() * (reference_log_probs - intervention_log_probs), dim=-1)
    reference_nll = F.nll_loss(reference_log_probs, labels, reduction='mean')
    intervention_nll = F.nll_loss(intervention_log_probs, labels, reduction='mean')
    return {
        'mean_kl': float(divergences.mean().item()),
        'max_kl': float(divergences.max().item()),
        'p95_kl': float(torch.quantile(divergences, 0.95).item()),
        'reference_nll': float(reference_nll.item()),
        'intervention_nll': float(intervention_nll.item()),
        'delta_nll': float((intervention_nll - reference_nll).item()),
    }

In [ ]:
candidate = manifest['receiver_heads'][0]
candidate_layer, candidate_head = candidate['layer'], candidate['query_head']
causal_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=torch.float16, attn_implementation='sdpa'
).to('cuda').eval()
causal_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
causal_samples = []
mask_sanity_max_abs_logit = None
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

for sample_index, (prompt, artifact_path) in enumerate(zip(PROMPTS, artifact_paths, strict=True)):
    causal_model.set_attn_implementation('sdpa')
    rendered = render_prompt(causal_tokenizer, prompt)
    inputs = causal_tokenizer(rendered, return_tensors='pt')
    prompt_tokens = int(inputs['input_ids'].shape[-1])
    device_inputs = {key: value.to('cuda') for key, value in inputs.items()}
    with torch.inference_mode():
        generated = causal_model.generate(
            **device_inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True,
            pad_token_id=causal_tokenizer.eos_token_id,
        )[:, :MAX_SEQUENCE_LENGTH]
    generated_token_ids = generated[0, prompt_tokens:].tolist()
    eos_ids = causal_model.generation_config.eos_token_id
    eos_ids = {eos_ids} if isinstance(eos_ids, int) else set(eos_ids or [])
    if not generated_token_ids or generated_token_ids[-1] not in eos_ids:
        raise RuntimeError(f'math-{sample_index:03d} did not reproduce a completed trace.')
    reproduced_text, _ = decoded_token_offsets(causal_tokenizer, generated_token_ids)
    if reproduced_text != run_summaries[sample_index]['generated_text']:
        raise RuntimeError(f'math-{sample_index:03d} generation was not deterministic.')
    query_end = int(generated.shape[-1]) - 1
    trace = load_trace(artifact_path)
    starts, ends, texts = trace['span_starts'], trace['span_ends'], trace['span_text']
    raw_attention = trace['vertical_scores'][candidate_layer, candidate_head].astype(np.float64)
    normalized_attention = cross_head_normalized_scores(
        trace['vertical_scores'][candidate_layer], candidate_head
    )
    all_head_attention = trace['vertical_scores'].mean(axis=(0, 1)).astype(np.float64)
    evaluation_start = int(np.max(ends))
    if evaluation_start >= query_end:
        raise RuntimeError('Candidate spans leave no common downstream evaluation window.')
    logit_start, logit_end = evaluation_start - 1, query_end - 1
    labels = generated[0, evaluation_start:query_end]
    causal_model.set_attn_implementation('eager')
    base_mask = causal_mask_tensor(int(generated.shape[-1]), query_end=query_end, dtype=causal_model.dtype)
    with torch.inference_mode():
        reference_output = causal_model(
            input_ids=generated, attention_mask=base_mask, use_cache=False, return_dict=True
        )
    reference_logits = reference_output.logits[0, logit_start:logit_end].detach()
    del reference_output, base_mask
    if sample_index == 0:
        with torch.inference_mode():
            standard_output = causal_model(
                input_ids=generated, attention_mask=torch.ones_like(generated), use_cache=False, return_dict=True
            )
        standard_logits = standard_output.logits[0, logit_start:logit_end]
        mask_sanity_max_abs_logit = float((reference_logits - standard_logits).abs().max().item())
        if mask_sanity_max_abs_logit > 5e-3:
            raise RuntimeError(f'Custom 4D base mask changed reference logits by {mask_sanity_max_abs_logit}.')
        del standard_output, standard_logits
    span_results = []
    for span_index in range(len(starts)):
        blocked = (int(starts[span_index]), int(ends[span_index]))
        mask = causal_mask_tensor(
            int(generated.shape[-1]), blocked_span=blocked, query_end=query_end, dtype=causal_model.dtype
        )
        with torch.inference_mode():
            intervention_output = causal_model(
                input_ids=generated, attention_mask=mask, use_cache=False, return_dict=True
            )
        metrics = intervention_metrics(
            reference_logits, intervention_output.logits[0, logit_start:logit_end], labels
        )
        metrics.update({
            'span_index': int(span_index), 'token_start': blocked[0], 'token_end': blocked[1],
            'span_tokens': blocked[1] - blocked[0], 'text': str(texts[span_index]),
        })
        metrics.update({
            'raw_attention': float(raw_attention[span_index]),
            'cross_head_normalized_attention': float(normalized_attention[span_index]),
            'all_head_mean_attention': float(all_head_attention[span_index]),
        })
        span_results.append(metrics)
        del intervention_output, mask
    causal_scores = np.asarray([row['mean_kl'] for row in span_results])
    interior_indices = np.arange(1, len(starts) - 1)
    diagnostics = {
        'raw_attention': ranking_diagnostics(raw_attention, causal_scores),
        'cross_head_normalized': ranking_diagnostics(normalized_attention, causal_scores),
        'all_head_mean': ranking_diagnostics(all_head_attention, causal_scores),
        'raw_attention_interior': ranking_diagnostics(raw_attention, causal_scores, interior_indices),
        'cross_head_normalized_interior': ranking_diagnostics(normalized_attention, causal_scores, interior_indices),
    }
    selector_indices = {
        'raw_attention': diagnostics['raw_attention']['proxy_top_indices'][0],
        'cross_head_normalized': diagnostics['cross_head_normalized']['proxy_top_indices'][0],
        'all_head_mean': diagnostics['all_head_mean']['proxy_top_indices'][0],
        'recency': len(starts) - 1,
        'random': int(np.random.default_rng(SEED + sample_index).integers(len(starts))),
        'causal_oracle': int(np.argmax(causal_scores)),
    }
    causal_samples.append({
        'sample_id': f'math-{sample_index:03d}',
        'evaluation_token_start': evaluation_start, 'evaluation_token_end': query_end,
        'evaluation_tokens': query_end - evaluation_start, 'spans': span_results,
        'diagnostics': diagnostics, 'selector_indices': selector_indices,
        'selector_mean_kl': {name: float(causal_scores[index]) for name, index in selector_indices.items()},
    })
    del reference_logits, generated, labels, device_inputs
    gc.collect()
    torch.cuda.empty_cache()

diagnostic_names = tuple(causal_samples[0]['diagnostics'])
selector_names = tuple(causal_samples[0]['selector_mean_kl'])
aggregate = {
    'diagnostics': {
        name: {metric: float(np.mean([sample['diagnostics'][name][metric] for sample in causal_samples]))
               for metric in ('spearman', 'top_k_overlap', 'top_k_regret', 'top_1_regret')}
        for name in diagnostic_names
    },
    'selector_mean_kl': {
        name: float(np.mean([sample['selector_mean_kl'][name] for sample in causal_samples]))
        for name in selector_names
    },
}
causal_report = {
    'schema_version': 2, 'method': 'all_candidate_teacher_forced_all_head_sentence_suppression',
    'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'receiver_head': candidate, 'samples': causal_samples, 'aggregate': aggregate,
    'mask_sanity_max_abs_logit': mask_sanity_max_abs_logit,
    'peak_gpu_gib': torch.cuda.max_memory_allocated() / 1024**3,
    'limitations': ['three-prompt pilot', 'teacher-forced replay', 'sentence suppression broadcasts across all heads', 'receiver head selected and evaluated on the same traces'],
}
causal_path = output_dir / 'causal-results.json'
causal_path.write_text(json.dumps(causal_report, indent=2, sort_keys=True) + '\n', encoding='utf-8')
del causal_model
gc.collect()
torch.cuda.empty_cache()
causal_report

In [ ]:
selector_labels = ['Raw head', 'Cross-head norm.', 'All-head mean', 'Recency', 'Random', 'Oracle']
selector_scores = [aggregate['selector_mean_kl'][name] for name in selector_names]
plt.figure(figsize=(9, 4))
plt.bar(selector_labels, selector_scores)
plt.ylabel('Mean downstream KL of selected sentence')
plt.title('Selector performance against the causal oracle')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(output_dir / 'selector-causal-kl.png', dpi=160)
plt.show()

plt.figure(figsize=(7, 5))
for sample in causal_samples:
    x = np.asarray([row['raw_attention'] for row in sample['spans']])
    y = np.asarray([row['mean_kl'] for row in sample['spans']])
    plt.scatter(x + 1e-12, y + 1e-12, alpha=0.75, label=sample['sample_id'])
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Raw receiver-head attention')
plt.ylabel('Measured causal KL')
plt.title('Attention salience versus causal importance')
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / 'attention-causal-scatter.png', dpi=160)
plt.show()

## Download the run

Keep the downloaded ZIP. It contains corrected traces, run provenance, the receiver-head manifest, all-candidate causal ranking metrics, and all plots.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/anchorkv-colab-traces', 'zip', output_dir)
files.download(archive)